# AlphaEvolve + **Qwen3-VL-2B-Instruct** on **ChartQA** — Google Colab

Runs **AlphaEvolve** (`evolve_chartqa.py`) with fitness from **HuggingFaceM4/ChartQA** `val` via `evaluate_chartqa.py`. The search mutates the `EVOLVE-BLOCK` in `seed_instruct.py`; **Qwen/Qwen3-VL-2B-Instruct** weights stay fixed.

**Before / after** = generation-0 program vs best evolved program (`evolved_chartqa.py`).

**Note:** Train/holdout use the same *query counts* as the CharXiv evolution driver (102+51 / 26+13) mapped to consecutive ChartQA val slices — see `evaluate_chartqa.py`.

**Colab compute:** **Runtime → Change runtime type → GPU** (T4 or better).

**Prerequisites:** HF token (Qwen + dataset), `GEMINI_API_KEY`, GPU (T4+ recommended), project at `/content/evolving-vlm-inference-main`.

In [1]:
# --- Configuration ---
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/evolving-vlm-inference-main")
GIT_REPO_URL = os.environ.get("GIT_REPO_URL", "")

NUM_GENERATIONS = int(os.environ.get("NUM_GENERATIONS", "3"))
POPULATION_SIZE = int(os.environ.get("POPULATION_SIZE", "6"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("NUM_GENERATIONS:", NUM_GENERATIONS, "POPULATION_SIZE:", POPULATION_SIZE)

PROJECT_ROOT: /content/evolving-vlm-inference-main
NUM_GENERATIONS: 3 POPULATION_SIZE: 6


In [2]:
%%capture
!pip install -q "transformers>=4.37.0" qwen-vl-utils torch torchvision pillow accelerate google-generativeai bitsandbytes datasets huggingface_hub

In [3]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. In Colab: Runtime → Change runtime type → GPU (e.g. T4), then re-run."
    )
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-80GB


In [4]:
import zipfile
from pathlib import Path

zip_path = Path("/content/evolving-vlm-inference-main.zip")  # change if your file name differs
out_dir = Path("/content")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(out_dir)

# If the zip already contains a top-level folder "evolving-vlm-inference-main", you're done.
# If it extracted into a wrong name, rename so this path exists:
assert (out_dir / "evolving-vlm-inference-main").is_dir(), list(out_dir.iterdir())

In [5]:
import os, shutil, subprocess
from pathlib import Path

if GIT_REPO_URL:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", "--depth", "1", GIT_REPO_URL, str(PROJECT_ROOT)], check=True)
elif not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"{PROJECT_ROOT} not found. Upload evolving-vlm-inference-main to /content or set GIT_REPO_URL."
    )

os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())

cwd: /content/evolving-vlm-inference-main


In [6]:
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import login
    login()

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Enter GEMINI_API_KEY: ")

assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY is required."

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Enter GEMINI_API_KEY: ··········


In [7]:
import os
import sys
import importlib.util
from pathlib import Path

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m


from datasets import load_dataset

_sample_dir = Path("/tmp/chartqa_colab_sample")
_sample_dir.mkdir(parents=True, exist_ok=True)
_ds = load_dataset("HuggingFaceM4/ChartQA", split="val")
_row = _ds[0]
_img = _row["image"]
_p = _sample_dir / "sample0.png"
if hasattr(_img, "convert"):
    _img.convert("RGB").save(_p)
_lab = _row.get("label")
if isinstance(_lab, (list, tuple)):
    _gold = str(_lab[0]) if _lab else ""
else:
    _gold = str(_lab)

SAMPLE = {
    "image_path": str(_p),
    "question": _row["query"],
    "qid": None,
    "gold": _gold,
}
print("Sample question:", SAMPLE["question"][:200])
print("Gold:", SAMPLE["gold"])

README.md:   0%|          | 0.00/852 [00:00<?, ?B/s]

data/train-00000-of-00003-49492f364babfa(…):   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00001-of-00003-7302bae5e425bb(…):   0%|          | 0.00/311M [00:00<?, ?B/s]

data/train-00002-of-00003-194c9400785577(…):   0%|          | 0.00/315M [00:00<?, ?B/s]

data/val-00000-of-00001-0f11003c77497969(…):   0%|          | 0.00/50.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e2cd0b7a0f9eb20(…):   0%|          | 0.00/68.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/28299 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1920 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Sample question: What's the color of graph with 56 as the highest value?
Gold: Blue


In [8]:
# --- Generation 0 (initial seed program) ---
initial = load_module("alphaevolve_seed_colab", PROJECT_ROOT / "seed_instruct.py")
before_pred = initial.vlm_inference(
    SAMPLE["image_path"], SAMPLE["question"], SAMPLE["qid"]
)
print("=== EXAMPLE — generation 0 (seed_instruct.py) ===")
print("Prediction:", before_pred)
print("Gold      :", SAMPLE["gold"])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== EXAMPLE — generation 0 (seed_instruct.py) ===
Prediction: The graph with 56 as the highest value is green.
Gold      : Blue


In [9]:
from evaluate_chartqa import evaluate

print("=== BENCHMARK — generation 0 (seed_instruct.py) ===")
bench_before = evaluate(initial, return_details=False)
for k in (
    "desc_accuracy",
    "num_desc_evaluated",
    "num_errors",
    "total_time",
    "avg_time_per_query",
):
    print(f"  {k}: {bench_before[k]}")

=== BENCHMARK — generation 0 (seed_instruct.py) ===
  desc_accuracy: 0.005208333333333333
  num_desc_evaluated: 192
  num_errors: 0
  total_time: 874.5328071117401
  avg_time_per_query: 4.554858370373647


In [10]:
# Free GPU memory before evolution
import gc
import sys
import torch

_mname = "alphaevolve_seed_colab"
if _mname in sys.modules:
    mod = sys.modules.pop(_mname)
    for attr in ("_model", "_processor"):
        if hasattr(mod, attr):
            setattr(mod, attr, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released generation-0 module from GPU memory.")

Released generation-0 module from GPU memory.


In [11]:
import evolve_chartqa

evolve_chartqa.NUM_GENERATIONS = NUM_GENERATIONS
evolve_chartqa.POPULATION_SIZE = POPULATION_SIZE

population, history = evolve_chartqa.run_evolution()
print("History entries:", len(history))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


AlphaEvolve — Qwen3-VL-2B-Instruct (ChartQA)
Generations : 25  |  Population : 10
Train split : 102 desc + 51 rsn samples
Holdout     : 26 desc + 13 rsn samples
Fitness     : descriptive accuracy (exact-match, train split)
Saves best  : by holdout accuracy (anti-overfitting)
Crossover   : every 5 generations
Seeds       : seed_instruct.py + 0 extra variants  →  evolved_chartqa.py

[Gen 0] Evaluating seed: seed_instruct.py


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

  Train acc  : 0.7%  |  Holdout acc : 0.0%  |  4.387s/q

  Best gen-0 seed by holdout: 0.0%  (train 0.7%)  → saved to evolved_chartqa.py

[Gen 1/25]
  Parent: 0.7%/0.0% acc (gen 0)


  [RETRY 1] Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. 
  [RETRY 2] Gemini error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


...

  [RETRY 3] Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. 
  Mutation failed — skipping generation.

[Gen 24/25]
  Parent: 0.7%/0.0% acc (gen 0)


  [RETRY 1] Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. 
  [RETRY 2] Gemini error: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


  [RETRY 3] Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. 
  Mutation failed — skipping generation.

[Gen 25/25]
  Parent: 0.7%/0.0% acc (gen 0)


  [RETRY 1] Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. 
  [RETRY 2] Gemini error: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


  [RETRY 3] Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. 
  Mutation failed — skipping generation.

Evolution complete!
  Best by train   : 0.7%  holdout=0.0%
  Best by holdout : 0.0%  train=0.7%  ← saved to disk
  Best speed      : 0.7%  (4.387s/q)
  Best overall    : 0.7%  score=0.0000
  Log             : evolution_log_chartqa.json
History entries: 26


In [12]:
# --- Best program after AlphaEvolve (evolved_chartqa.py) ---
evolved = load_module("evolved_chartqa_colab", PROJECT_ROOT / "evolved_chartqa.py")
after_pred = evolved.vlm_inference(
    SAMPLE["image_path"], SAMPLE["question"], SAMPLE["qid"]
)
print("=== EXAMPLE — after AlphaEvolve (evolved_chartqa.py) ===")
print("Prediction:", after_pred)
print("Gold      :", SAMPLE["gold"])

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

=== EXAMPLE — after AlphaEvolve (evolved_chartqa.py) ===
Prediction: The graph with 56 as the highest value is green.
Gold      : Blue


In [13]:
from evaluate_chartqa import evaluate

print("=== BENCHMARK — after AlphaEvolve (evolved_chartqa.py) ===")
bench_after = evaluate(evolved, return_details=False)
for k in (
    "desc_accuracy",
    "num_desc_evaluated",
    "num_errors",
    "total_time",
    "avg_time_per_query",
):
    print(f"  {k}: {bench_after[k]}")

print("\n=== SUMMARY (ChartQA: seed_instruct → evolved_chartqa) ===")
print(f"Accuracy: {bench_before['desc_accuracy']:.4f} → {bench_after['desc_accuracy']:.4f}")

=== BENCHMARK — after AlphaEvolve (evolved_chartqa.py) ===
  desc_accuracy: 0.005208333333333333
  num_desc_evaluated: 192
  num_errors: 0
  total_time: 847.0765717029572
  avg_time_per_query: 4.411857144286235

=== SUMMARY (ChartQA: seed_instruct → evolved_chartqa) ===
Accuracy: 0.0052 → 0.0052
